# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# EDA & Kiểm tra split (Bước 0)
from dataset import load_split, check_split, NUM_CLASSES, CLASS_NAMES
import matplotlib.pyplot as plt
import pandas as pd
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
stats = check_split(train_df, val_df, test_df, IMAGES_DIR)
print('Thống kê kiểm tra split:', stats['n'])
print('Overlap giao giữa các tập:', stats['overlap'])
df_counts = pd.DataFrame(stats['per_class'])
df_counts.index = [CLASS_NAMES[i] for i in df_counts.index]
print(df_counts)
df_counts.plot(kind='bar', figsize=(10, 4), rot=45)
plt.title('Phân bố 9 lớp DeepWeeds (Fold 0)')
plt.ylabel('Số lượng ảnh')
plt.tight_layout()
plt.show()


In [ ]:
# Kiểm tra pipeline sanity check (Checklist gỡ lỗi)
from train import set_seed
from model import build_model
from losses import FocalLoss
import torch, torch.nn as nn
set_seed(42)
# 1. Loss ban đầu xấp xỉ ln(9) = 2.197
dummy_model = build_model('resnet50', pretrained=True, num_classes=9)
dummy_x = torch.randn(8, 3, 224, 224)
dummy_y = torch.randint(0, 9, (8,))
init_out = dummy_model(dummy_x)
init_loss = nn.CrossEntropyLoss()(init_out, dummy_y).item()
print(f'Mất mát CE ban đầu: {init_loss:.4f} (kỳ vọng ~2.197)')
# 2. Overfit 1 batch nhỏ
opt = torch.optim.Adam(dummy_model.parameters(), lr=1e-3)
for _ in range(50):
    opt.zero_grad()
    l = nn.CrossEntropyLoss()(dummy_model(dummy_x), dummy_y)
    l.backward(); opt.step()
print(f'Loss sau khi overfit 1 batch: {l.item():.6f} (gần 0: đạt)')


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Bước 1: So sánh >= 5 Backbone
from train import Config, run
from model import build_model, count_params, count_gmacs
backbones = [
    ('B01', 'resnet50'),
    ('B02', 'resnext50_32x4d'),
    ('B03', 'convnext_tiny'),
    ('B04', 'swin_tiny_patch4_window7_224'),
    ('B05', 'mobilenetv3_large_100')
]
print('Danh sách 5 backbone sẵn sàng cho thực nghiệm Bước 1.')


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Bước 2: Tối ưu hoá công thức huấn luyện trên ConvNeXt-Tiny
# Trục A: Khởi tạo (frozen, scratch, finetune)
# Trục B: Augmentation (basic, randaug, cutmix)
# Trục C: Loss (ce, focal, label_smoothing, class_weighted)
# Trục F: Regularization (EMA)
# T09: Kết hợp CutMix + Label Smoothing + EMA
print('Các cấu hình T01 -> T09 được định nghĩa theo nguyên tắc N1 (1 thay đổi mỗi lần).')


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Bước 3: Phương pháp suy luận & đo độ trễ
from inference import predict_logits, aggregate_views, fit_temperature, apply_temperature, ensemble_probs, fuse_conv_bn
from benchmark import bench, latency_report, tta_latency
print('Kiểm tra module benchmark & inference: sẵn sàng.')


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Bước 4: Chung kết 3 seeds (F01) và Mốc (T00)
print('Chạy đánh giá với 3 seed (0, 1, 2) cho F01 và T00, lưu dự đoán test fold 0.')


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Bước 5: Kiểm tra hoàn tất các sản phẩm results.xlsx, curves/, report.md
import os
assert os.path.exists('results.xlsx') or os.path.exists('../results.xlsx')
print('Đã hoàn tất toàn bộ sản phẩm!')
